# Node Classification with PyTorch Geometric in TopologicPy

The goal is to predict the **room type** of each node in a graph.

## What this notebook assumes

You already have a prepared dataset directory containing:

- `graphs.csv`
- `nodes.csv`
- `edges.csv`

## What this notebook does

1. Reads the dataset from disk
2. Verifies the CSV schema
3. Loads the dataset into `topologicpy.PyG`
4. Trains a baseline **GraphSAGE** model for **node classification**
5. Evaluates the model on validation and test nodes
6. Visualizes training history and confusion matrices
7. Exports node-level predictions


In [ ]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Imports

We use standard Python tools for file handling and tables, and `topologicpy.PyG` for the graph machine learning workflow.

In [ ]:
from pathlib import Path
import json
import pandas as pd

from topologicpy.PyG import PyG
from topologicpy.Helper import Helper

## 2. Check the TopologicPy Version

In [ ]:
print("This tutorial requires topologicpy version 0.9.31 or newer.")
print(Helper.Version())

## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [ ]:
renderer = "vscode"

## Baseline
```
# ------------------------------------------------------------------
# DATASET LOCATION
# ------------------------------------------------------------------
# Change this path if your prepared clean dataset is elsewhere.
# The folder must contain graphs.csv, nodes.csv, and edges.csv.
DATASET_PATH = Path(r"C:\Users\sarwj\OneDrive - Cardiff University\IAAC\2025-26\S3 - Buildings As Graphs\notebooks\Supporting Files\dataset_node_classification").resolve()
VIS_PATH = Path(r"C:\Users\sarwj\OneDrive - Cardiff University\IAAC\2025-26\S3 - Buildings As Graphs\notebooks\Supporting Files\dataset_for_visualisation").resolve()

# ------------------------------------------------------------------
# TASK DEFINITION
# ------------------------------------------------------------------
PREDICTION_LEVEL = "node"               # Prediction target level: graph | node | edge | link.
TASK = "classification"                 # Learning task type: classification | regression | link_prediction.
GRAPH_LABEL_TYPE = "categorical"        # Type of graph-level label: continuous for regression targets such as cooling/heating load.
NODE_LABEL_TYPE = "categorical"         # Type of node-level label: categorical class index, e.g. room type or element type.
EDGE_LABEL_TYPE = "categorical"         # Type of edge-level label: categorical class index; currently not used in this setup.
HOLDOUT_GROUP_BY = "label"              # Grouping key for holdout splitting; keeps items with the same label together where supported.


# ------------------------------------------------------------------
# MODEL DEFINITION
# ------------------------------------------------------------------
CONV = "sage"                           # Graph convolution operator to use; "sage" selects GraphSAGE.
HIDDEN_DIMS = (64, 64, 64)              # Hidden layer dimensions; two values define two hidden graph-convolution layers.
ACTIVATION = "relu"                     # Activation function used between hidden layers.
DROPOUT = 0.0                           # Dropout probability applied during training to reduce overfitting.
BATCH_NORM = True                       # Whether to apply batch normalisation after hidden layers.
RESIDUAL = False                        # Whether to use residual/skip connections between compatible layers.
POOLING = "mean"                        # Graph-level pooling operation used to aggregate node embeddings into graph embeddings.


# ------------------------------------------------------------------
# DATA SPLIT SETTINGS
# ------------------------------------------------------------------
CROSS_VALIDATION = "holdout"            # Validation strategy; "holdout" uses one train/validation/test split.
TRAIN_RATIO = 0.7                       # Proportion of data assigned to the training set.
VAL_RATIO = 0.15                        # Proportion of data assigned to the validation set.
TEST_RATIO = 0.15                       # Proportion of data assigned to the test set.
RANDOM_STATE = 42                       # Random seed used for reproducible splitting, shuffling, and training initialisation where supported.
SHUFFLE = True                          # Whether to shuffle the dataset before splitting.


# ------------------------------------------------------------------
# TRAINING SETTINGS
# ------------------------------------------------------------------
LEARNING_RATE = 1e-2                    # Optimiser learning rate.
BATCH_SIZE = 10                         # Number of graphs/samples per training batch.
EPOCHS = 50                             # Maximum number of training epochs.
OPTIMIZER = "adamw"                     # Optimiser to use for training; AdamW includes decoupled weight decay.
WEIGHT_DECAY = 0.01                     # L2 regularisation strength used by the optimiser.
GRADIENT_CLIP_NORM = 1.0                # Maximum gradient norm for clipping; helps stabilise training.
EARLY_STOPPING = True                   # Whether to stop training early when validation performance stops improving.
EARLY_STOPPING_PATIENCE = 12            # Number of epochs without validation improvement before early stopping is triggered.
USE_GPU = True                          # Whether to use a CUDA-capable GPU when available.
```

In [ ]:
# ------------------------------------------------------------------
# DATASET LOCATION
# ------------------------------------------------------------------
# Change this path if your prepared clean dataset is elsewhere.
# The folder must contain graphs.csv, nodes.csv, and edges.csv.
DATASET_PATH = Path(r"C:\Users\sarwj\OneDrive - Cardiff University\IAAC\2025-26\S3 - Buildings As Graphs\notebooks\Supporting Files\dataset_node_classification").resolve()
VIS_PATH = Path(r"C:\Users\sarwj\OneDrive - Cardiff University\IAAC\2025-26\S3 - Buildings As Graphs\notebooks\Supporting Files\dataset_for_visualisation").resolve()

# ------------------------------------------------------------------
# TASK DEFINITION
# ------------------------------------------------------------------
PREDICTION_LEVEL = "node"               # Prediction target level: graph | node | edge | link.
TASK = "classification"                 # Learning task type: classification | regression | link_prediction.
GRAPH_LABEL_TYPE = "categorical"        # Type of graph-level label: continuous for regression targets such as cooling/heating load.
NODE_LABEL_TYPE = "categorical"         # Type of node-level label: categorical class index, e.g. room type or element type.
EDGE_LABEL_TYPE = "categorical"         # Type of edge-level label: categorical class index; currently not used in this setup.
HOLDOUT_GROUP_BY = "label"              # Grouping key for holdout splitting; keeps items with the same label together where supported.


# ------------------------------------------------------------------
# MODEL DEFINITION
# ------------------------------------------------------------------
CONV = "sage"                           # Graph convolution operator to use; "sage" selects GraphSAGE.
HIDDEN_DIMS = (64, 64, 64)              # Hidden layer dimensions; two values define two hidden graph-convolution layers.
ACTIVATION = "relu"                     # Activation function used between hidden layers.
DROPOUT = 0.0                           # Dropout probability applied during training to reduce overfitting.
BATCH_NORM = True                       # Whether to apply batch normalisation after hidden layers.
RESIDUAL = False                        # Whether to use residual/skip connections between compatible layers.
POOLING = "mean"                        # Graph-level pooling operation used to aggregate node embeddings into graph embeddings.


# ------------------------------------------------------------------
# DATA SPLIT SETTINGS
# ------------------------------------------------------------------
CROSS_VALIDATION = "holdout"            # Validation strategy; "holdout" uses one train/validation/test split.
TRAIN_RATIO = 0.7                       # Proportion of data assigned to the training set.
VAL_RATIO = 0.15                        # Proportion of data assigned to the validation set.
TEST_RATIO = 0.15                       # Proportion of data assigned to the test set.
RANDOM_STATE = 42                       # Random seed used for reproducible splitting, shuffling, and training initialisation where supported.
SHUFFLE = True                          # Whether to shuffle the dataset before splitting.


# ------------------------------------------------------------------
# TRAINING SETTINGS
# ------------------------------------------------------------------
LEARNING_RATE = 1e-2                    # Optimiser learning rate.
BATCH_SIZE = 10                         # Number of graphs/samples per training batch.
EPOCHS = 50                             # Maximum number of training epochs.
OPTIMIZER = "adamw"                     # Optimiser to use for training; AdamW includes decoupled weight decay.
WEIGHT_DECAY = 0.01                     # L2 regularisation strength used by the optimiser.
GRADIENT_CLIP_NORM = 1.0                # Maximum gradient norm for clipping; helps stabilise training.
EARLY_STOPPING = True                   # Whether to stop training early when validation performance stops improving.
EARLY_STOPPING_PATIENCE = 12            # Number of epochs without validation improvement before early stopping is triggered.
USE_GPU = True                          # Whether to use a CUDA-capable GPU when available.

## 5. Check the dataset files

Before loading the dataset into PyG, it is good practice to verify that the three required CSV files exist.

In [ ]:
required_files = [
    DATASET_PATH / "graphs.csv",
    DATASET_PATH / "nodes.csv",
    DATASET_PATH / "edges.csv",
]

for f in required_files:
    print(f"{f.name}: {'FOUND' if f.exists() else 'MISSING'} -> {f}")

if not all(f.exists() for f in required_files):
    raise FileNotFoundError(
        "One or more required CSV files are missing. "
        "Check PREPARED_DATASET_DIR before proceeding."
    )

## 6. Inspect the CSV schema

This step is pedagogically useful because it makes the node-classification setup visible.

For node classification, the important pieces are:

### `nodes.csv`
- node identifiers
- node labels
- node features
- train/validation/test masks

### `edges.csv`
- source node
- destination node
- optional edge features

### `graphs.csv`
- graph identifier
- number of nodes

In [ ]:
graphs_df = pd.read_csv(DATASET_PATH / "graphs.csv")
nodes_df = pd.read_csv(DATASET_PATH / "nodes.csv")
edges_df = pd.read_csv(DATASET_PATH / "edges.csv")

print("graphs.csv shape:", graphs_df.shape)
print("nodes.csv shape:", nodes_df.shape)
print("edges.csv shape:", edges_df.shape)

In [ ]:
print("graphs.csv columns:")
print(list(graphs_df.columns))
print()

print("nodes.csv columns:")
print(list(nodes_df.columns))
print()

print("edges.csv columns:")
print(list(edges_df.columns))

In [ ]:
feature_cols = [c for c in nodes_df.columns if c.startswith("feat_")]
mask_cols = [c for c in nodes_df.columns if c.endswith("_mask")]

summary = {
    "num_graphs": int(graphs_df["graph_id"].nunique()),
    "num_nodes": int(len(nodes_df)),
    "num_edges": int(len(edges_df)),
    "num_classes": int(nodes_df["label"].nunique()),
    "num_node_features": int(len(feature_cols)),
    "feature_columns": feature_cols,
    "mask_columns": mask_cols,
    "train_nodes": int(nodes_df["train_mask"].sum()) if "train_mask" in nodes_df.columns else None,
    "val_nodes": int(nodes_df["val_mask"].sum()) if "val_mask" in nodes_df.columns else None,
    "test_nodes": int(nodes_df["test_mask"].sum()) if "test_mask" in nodes_df.columns else None,
}

print(json.dumps(summary, indent=2))

## 7. Optional sanity checks

These checks help confirm that the prepared dataset really is ready for node classification.

In [ ]:
assert "label" in nodes_df.columns, "nodes.csv must contain a label column"
assert all(col in nodes_df.columns for col in ["train_mask", "val_mask", "test_mask"]), (
    "nodes.csv must contain train_mask, val_mask, and test_mask"
)
assert len(feature_cols) > 0, "No node feature columns found"

print("Unique labels:", sorted(nodes_df["label"].dropna().unique().tolist()))
print("Any missing labels?", bool(nodes_df["label"].isna().any()))
print("Any missing node features?", bool(nodes_df[feature_cols].isna().any().any()))

## 8. Load the dataset with TopologicPy/PyG

This is the key step where the prepared CSV files are converted into a PyG-ready dataset.

For **node classification**, the critical arguments are:

- `level="node"`
- `task="classification"`
- `nodeLabelType="categorical"`

The explicit masks in `nodes.csv` determine which nodes belong to the train, validation, and test subsets.

In [ ]:
# This dataset has graphs.csv with a categorical 'label' column -> graph classification.
pyg = PyG.ByCSVPath(
    path=str(DATASET_PATH),
    level=PREDICTION_LEVEL,
    task=TASK,
    graphLabelType=GRAPH_LABEL_TYPE,
    nodeLabelType=NODE_LABEL_TYPE,
    edgeLabelType=EDGE_LABEL_TYPE,

    # If your headers differ, override here (your attached CSVs match defaults):
    # graphIDHeader="graph_id", graphLabelHeader="label",
    # nodeIDHeader="node_id", nodeLabelHeader="label",
    # edgeSRCHeader="src_id", edgeDSTHeader="dst_id", edgeLabelHeader="label",
)

print("Dataset loaded successfully.")
print(json.dumps(pyg.Summary(), indent=2, default=str))

## 9. Define the baseline GraphSAGE model

This notebook uses a straightforward GraphSAGE baseline.

### Why GraphSAGE?
GraphSAGE is a strong default for node classification because it combines:

- node attributes
- neighborhood information
- inductive learning over graphs

### Why these hyperparameters?
These settings are intentionally simple and clear for a tutorial:

- convolution = GraphSAGE
- hidden dimensions = `(64, 64, 64)`
- activation = `relu`
- learning rate = `0.001`
- batch size = `32`
- epochs = `50`

In [ ]:
pyg.SetHyperparameters(
    # splitting / determinism
    cv=CROSS_VALIDATION,
    split=(TRAIN_RATIO, VAL_RATIO, TEST_RATIO),
    random_state=RANDOM_STATE,
    shuffle=SHUFFLE,

    # training
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    optimizer=OPTIMIZER,
    gradient_clip_norm=GRADIENT_CLIP_NORM,
    early_stopping=EARLY_STOPPING,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    use_gpu=USE_GPU,

    # model
    conv=CONV,
    hidden_dims=HIDDEN_DIMS,
    activation=ACTIVATION,
    dropout=DROPOUT,
    batch_norm=BATCH_NORM,
    residual=RESIDUAL,
    pooling=POOLING
)


print("Model and training hyperparameters have been set.")

## 10. Train the model

Training uses only the nodes where `train_mask == 1`.

The validation mask is used during training history tracking, and the test mask remains untouched until final evaluation.

In [ ]:
history = pyg.Train()

print("History keys:", list(history.keys()))
for k, v in history.items():
    if isinstance(v, (list, tuple)):
        print(f"{k}: {len(v)} values")
    else:
        print(f"{k}: {v}")

## 11. Plot the training history

This plot helps answer two common tutorial questions:

- Is the model learning?
- Is the validation curve diverging from the training curve?

In [ ]:
fig_hist = pyg.PlotHistory()
fig_hist.update_layout(width=900, height=700)
fig_hist.show()

## 12. Evaluate on validation and test nodes

Validation metrics help with model checking.

Test metrics are the final held-out results and should be the main numbers reported for this tutorial run.

In [ ]:
def pretty_print_metrics(title: str, metrics: dict) -> None:
    print(title)
    print("=" * 80)
    df = pd.DataFrame({"metric": list(metrics.keys()), "value": list(metrics.values())})
    df = df.sort_values("metric").reset_index(drop=True)
    print(df.to_string(index=False))
    print("=" * 80)

val_metrics = pyg.Validate()
test_metrics = pyg.Test()

pretty_print_metrics("Validation metrics", val_metrics)
pretty_print_metrics("Test metrics", test_metrics)

## 13. Plot confusion matrices

Confusion matrices are especially useful in node classification because they show **which room types are confused with which other room types**.

In [ ]:
fig_val_cm = pyg.PlotConfusionMatrix(
    split="val",
    normalize=False,
    title="Validation Confusion Matrix"
)
fig_val_cm.show()

In [ ]:
fig_test_cm = pyg.PlotConfusionMatrix(
    split="test",
    normalize=False,
    title="Test Confusion Matrix"
)
fig_test_cm.show()

## 14. Predict node labels for all nodes

This step attaches predictions to the internal PyG data objects.

It is useful when you want to inspect which nodes were correctly or incorrectly classified.

In [ ]:
_ = pyg.Predict(split="all", return_probs=True, attach_to_data=True)
print("Predictions attached to the dataset.")

## 15. Export node-level predictions to CSV

The exported table is useful for:

- error analysis
- joining predictions back to geometry
- reviewing results graph by graph

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

def _to_class_index(value):
    """
    Convert scalar / length-1 array / one-hot vector to an integer class index.
    """
    arr = np.asarray(value)
    arr = np.squeeze(arr)

    if arr.ndim == 0:
        return int(arr)

    if arr.ndim == 1:
        if arr.size == 1:
            return int(arr[0])
        return int(np.argmax(arr))

    raise ValueError(f"Cannot convert value with shape {arr.shape} to class index.")

def export_node_predictions(pyg_obj, output_csv: Path) -> pd.DataFrame:
    pred_report = pyg_obj.Predict(
        split="all",
        return_probs=True,
        attach_to_data=True
    )

    pred_by_graph = pred_report["pred"]
    y_true_by_graph = pred_report["y_true"]
    prob_by_graph = pred_report.get("prob", None)

    print("Number of graphs in pred_report['pred']:", len(pred_by_graph))
    print("Number of graphs in pyg_obj.data_list:", len(pyg_obj.data_list))

    rows = []

    for graph_idx, data in enumerate(pyg_obj.data_list):
        graph_id = int(data.graph_id.item()) if hasattr(data, "graph_id") else graph_idx
        n = data.num_nodes

        graph_pred = pred_by_graph[graph_idx]
        graph_true = y_true_by_graph[graph_idx]
        graph_prob = prob_by_graph[graph_idx] if prob_by_graph is not None else None

        graph_pred = np.asarray(graph_pred)
        graph_true = np.asarray(graph_true)
        if graph_prob is not None:
            graph_prob = np.asarray(graph_prob)

        print(
            f"graph {graph_idx}: "
            f"num_nodes={n}, pred_shape={graph_pred.shape}, true_shape={graph_true.shape}, "
            f"prob_shape={None if graph_prob is None else graph_prob.shape}"
        )

        train_mask = data.train_mask.detach().cpu().numpy() if hasattr(data, "train_mask") else None
        val_mask = data.val_mask.detach().cpu().numpy() if hasattr(data, "val_mask") else None
        test_mask = data.test_mask.detach().cpu().numpy() if hasattr(data, "test_mask") else None

        if len(graph_pred) != n:
            raise ValueError(
                f"Prediction length mismatch in graph {graph_idx}: "
                f"len(graph_pred)={len(graph_pred)} but num_nodes={n}"
            )
        if len(graph_true) != n:
            raise ValueError(
                f"Ground-truth length mismatch in graph {graph_idx}: "
                f"len(graph_true)={len(graph_true)} but num_nodes={n}"
            )

        for node_idx in range(n):
            y_true_i = _to_class_index(graph_true[node_idx])
            y_pred_i = _to_class_index(graph_pred[node_idx])

            row = {
                "graph_id": graph_id,
                "node_id": node_idx,
                "y_true": y_true_i,
                "y_pred": y_pred_i,
            }

            if graph_prob is not None:
                prob_i = np.asarray(graph_prob[node_idx]).squeeze()
                if prob_i.ndim == 1 and y_pred_i < prob_i.size:
                    row["y_pred_prob"] = float(prob_i[y_pred_i])

            if train_mask is not None:
                row["train_mask"] = bool(train_mask[node_idx])
            if val_mask is not None:
                row["val_mask"] = bool(val_mask[node_idx])
            if test_mask is not None:
                row["test_mask"] = bool(test_mask[node_idx])

            rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    return df

predictions_csv = DATASET_PATH / "node_predictions_baseline.csv"
predictions_df = export_node_predictions(pyg, predictions_csv)

print(f"Saved predictions to: {predictions_csv}")
display(predictions_df.head())

## 16. Compare Prediction vs. True visually

In [ ]:
from topologicpy.Graph import Graph

graphs = Graph.ByCSVPath(path=str(VIS_PATH))
print(graphs)

In [ ]:
from topologicpy.Dictionary import Dictionary
from topologicpy.Topology import Topology
from topologicpy.Color import Color
g = graphs[50]
verts = Graph.Vertices(g)
for v in verts:
    d = Topology.Dictionary(v)
    true = int(Dictionary.ValueAtKey(d, "true"))
    pred = int(Dictionary.ValueAtKey(d, "pred"))
    if not true == pred:
        size = 30
        true_color = "red"
        pred_color = "red"
    else:
        size = 14
        true_color = Color.ByValueInRange(true, minValue=0, maxValue=8)
        pred_color = Color.ByValueInRange(pred, minValue=0, maxValue=8)
    d = Dictionary.SetValuesAtKeys(d, ["true_color", "pred_color", "size", "true", "pred"], [true_color, pred_color, size, true, pred])
    v = Topology.SetDictionary(v, d)

g = Graph.Reshape(g)
Topology.Show(g, vertexSize=6, vertexSizeKey="size", vertexColorKey="true_color", showVertexLabel=True, vertexLabelKey="true", backgroundColor="white", camera=[0,0,3])
Topology.Show(g, vertexSize=6, vertexSizeKey="size", vertexColorKey="pred_color", showVertexLabel=True, vertexLabelKey="pred", backgroundColor="white", camera=[0,0,3])